In [ ]:
pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import (accuracy_score,
                             log_loss)
from sklearn.ensemble import \
     (RandomForestRegressor as RF,
      GradientBoostingRegressor as GBR)
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
import lightgbm as lgb
import timeit

In [2]:
df = pd.read_csv('CVX_Final_Result.csv', index_col=0, parse_dates=True)
df.head()
# Replace 'malignant' with 1 and 'benign' with 0
df = df.query("condition != 'black'")
# Multiple categories become 1
df['condition'] = df['condition'].replace({
    'neutral': 0,
    'positive': 1,
    'negative': 1
})
print(df['condition'].value_counts(dropna=False))

condition
1    161
0    138
Name: count, dtype: int64


/tmp/ipykernel_6611/1155899229.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('CVX_Final_Result.csv', index_col=0, parse_dates=True)
/tmp/ipykernel_6611/1155899229.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['condition'] = df['condition'].replace({


In [ ]:
df.head()

,X,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,skewness_raw,mean_raw,Mean_Tonic,Max_Tonic,Min_Tonic,SD_Tonic,Mean_Phasic,Max_Phasic,Min_Phasic,SD_Phasic
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
1,SUD_003_01_EDA_10s,0.356146,0.361270,0.354865,0.354865,0.353584,0.353584,0.352302,0.352302,0.351021,...,-3.042697,0.318193,-0.024601,0.358658,-0.362070,0.237248,0.343182,0.695387,0.000000,0.228815
2,SUD_003_01_EDA_20s,0.333086,0.330524,0.331805,0.329243,0.330524,0.329243,0.329243,0.329243,0.327962,...,-0.078770,0.321844,-0.161865,0.155737,-0.370690,0.176927,0.483110,0.701306,0.154895,0.183622
3,SUD_003_01_EDA_30s,0.307464,0.310026,0.308745,0.308745,0.308745,0.307464,0.307464,0.307464,0.307464,...,-0.193585,0.302211,0.268743,0.295805,0.169441,0.033962,0.033986,0.139749,0.006041,0.035939
4,SUD_003_01_EDA_40s,0.294653,0.294653,0.293372,0.293372,0.295934,0.292091,0.292091,0.292091,0.292091,...,-0.095485,0.287799,0.266628,0.272488,0.262885,0.003119,0.020351,0.027083,0.006031,0.006480
5,SUD_003_01_EDA_50s,0.279280,0.277999,0.276718,0.275436,0.276718,0.275436,0.275436,0.275436,0.272874,...,0.792911,0.271977,0.270931,0.273679,0.265406,0.002695,0.001705,0.005345,0.000305,0.001209


In [3]:
cols_to_scale = df.columns.difference(['condition', 'source_rows', 'X'])
X = df[cols_to_scale]

# 2. Define your target (y)
# Fixed the typo from 'condtion' to 'condition'
Y = df['condition']
#Split test and train prior to standardization to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

#Do not do log, do z score for all variables
from sklearn.preprocessing import StandardScaler

# 1. Get a list of all columns EXCEPT 'condition'
cols_to_scale = df.columns.difference(['condition', 'source_rows', 'X'])

# 2. Initialize and apply the scaler only to those columns
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

# 4. Transform TEST
# Use the mean and std from X_train to scale X_test
X_test = scaler.transform(X_test)

# Now df has standardized features, but 'condition' remains 0/1 (or its original state)

In [4]:
start_time = timeit.default_timer()

condition_col = 'condition'  # rename the variable to avoid collision
colnames = df.drop(columns=[condition_col, 'source_rows']).columns.tolist()
#Logistic Regression for identifying significant vectors, currently broken
formula = f'{condition_col} ~ {" + ".join(colnames)}'
model = smf.glm(formula=formula, data=df, family=sm.families.Binomial())

result = model.fit()
print(result.summary())

# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)


                 Generalized Linear Model Regression Results                  
Dep. Variable:              condition   No. Observations:                  253
Model:                            GLM   Df Residuals:                        0
Model Family:                Binomial   Df Model:                          252
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.9947e-09
Date:                Fri, 03 Jul 2026   Deviance:                   3.9898e-09
Time:                        12:37:57   Pearson chi2:                 1.99e-09
No. Iterations:                    24   Pseudo R-squ. (CS):             0.7479
Covariance Type:            nonrobust                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [5]:
# Define start time to calculate computation time
start_time = timeit.default_timer()


# Initialize the LightGBM model
model = lgb.LGBMClassifier(learning_rate=0.09, max_depth=-5, random_state=42)

# Train the model
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='logloss')

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
#Cross validation
scores = cross_val_score(model, X_train, y_train, cv=5)

print(f"Scores per fold: {scores}")
print(f"Mean Accuracy: {scores.mean():.2f}")

# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)
print(f"Accuracy on the test set: {accuracy:.2f}")

print(classification_report(y_test, y_pred, digits=3))


[LightGBM] [Info] Number of positive: 129, number of negative: 110
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000336 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3552
[LightGBM] [Info] Number of data points in the train set: 239, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.539749 -> initscore=0.159332
[LightGBM] [Info] Start training from score 0.159332
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
#Feature importance map for lightgbm model
feat_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(feat_imp)

         feature  importance
0   kurtosis_raw          61
1            t40          59
2        std_raw          55
3      Max_Tonic          39
4    Mean_Phasic          37
5       SD_Tonic          35
6   skewness_raw          34
7      SD_Phasic          33
8     Min_Phasic          32
9             t5          28
10           t32          27
11       Min_raw          25
12           t39          22
13    Max_Phasic          21
14     Min_Tonic          19
15           t12          19
16           t28          18
17           t26          17
18    Mean_Tonic          16
19           t38          15
20           t13          15
21           t17          13
22           t35          13
23           t30          12
24           t25          12
25           t20          12
26            t7          11
27            t6          10
28            t3          10
29       Max_raw           9
30           t37           9
31            t8           9
32           t24           9
33           t